# Konu 10 — Zincirleme: üretim hattı

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Hücreler yukarıdakilerin oluşturduğu değişkenleri kullanır; birini atlarsan hata alırsın.

Konu 5'te slogan yazdırdın, Konu 8'de görselleri koddan ve toplu olarak ürettin; ikisi ayrı defterlerdeydi. Bugün onları
tek bir **hatta** bağlıyorsun: `ayarlar.json`'daki brif → metin modeliyle sloganlar → görsel modeliyle her slogana bir görsel.
Hattın her adımı bir **halka**; bir halkanın çıktısı sıradakinin girdisi.

Isınma, Adım 1 ve Adım 2 anahtarsız; Adım 3 ve sonrası `anahtar.txt` ister. Defterin tamamı 4 metin ve 3 görsel isteği
gönderir (Bonus 2 ile 4 görsel). Adım 6'da bilerek reddedilen 3 görsel isteği de var. Hepsi ücretsiz günlük kotanın küçük
bir kısmını tutar.

## Isınma — Hat kâğıt üstünde

Önce sahte halkalarla dene: üç küçük fonksiyon, hiçbiri internete çıkmıyor. Bu hücre ekrana bir şey yazmaz, yalnızca
fonksiyonları tanımlar.

In [ ]:
def slogan_yaz(konu):
    return konu + ": burada zaman yavaşlar"

def yonerge_kur(slogan, stil):
    return stil + " poster, idea: " + slogan

def gorsel_ciz(yonerge):
    return "[görsel] " + yonerge

Sıra zincirde: her satır bir öncekinin sonucunu alıyor.

In [ ]:
konu = "sessiz bir çalışma kafesi"
slogan = slogan_yaz(konu)
yonerge = yonerge_kur(slogan, "risograph")
gorsel = gorsel_ciz(yonerge)

print(slogan)
print(yonerge)
print(gorsel)

**Tahmin et, sohbete yaz:** ilk satırdaki konuyu `"gece açık bir kitapçı"` yapıp yeniden çalıştırırsan çıktının üç
satırından kaçı değişir?

Aşağıdaki hücre üsttekinin kopyası, yalnızca ilk satırı farklı. Çalıştır, tahminini kontrol et.

In [ ]:
konu = "gece açık bir kitapçı"
slogan = slogan_yaz(konu)
yonerge = yonerge_kur(slogan, "risograph")
gorsel = gorsel_ciz(yonerge)

print(slogan)
print(yonerge)
print(gorsel)

Üç satırın üçü de değişti. Hattın başındaki tek girdi sonuna kadar taşınıyor: başta yanlış bir kelime, sonda yanlış bir görsel.
Bu yüzden hattın en önemli satırı brif. Gerçek bir hatta her halka modele giden bir istektir; aşağıda sahte halkaların
yerini gerçek modeller alacak.

## Adım 1 — Ayarlar dosyası: `ayarlar.json`

Bu klasördeki `ayarlar.json` dosyasını VS Code'da aç: hattın bütün ayarları orada, koddan ayrı duruyor. Böyle bir dosyaya
**yapılandırma dosyası** (config) denir. Müşteri ya da konu değişince kodu değil, bu dosyayı değiştirirsin.

Dosyanın biçimi JSON, içeriği tek katmanlı bir sözlük. Konu 8'de günlüğü `json.dump` ile dosyaya yazdın; `json.load` bunun
tersini yapar: dosyayı okur, Python sözlüğüne çevirir. Konu 9'da video günlüğünü de böyle geri okudun.

In [ ]:
import json

with open("ayarlar.json", encoding="utf-8") as dosya:
    ayarlar = json.load(dosya)

print(ayarlar)

Sonra her ayarı kısa adlı bir değişkene al; aşağıdaki hücreler bu adları kullanacak:

In [ ]:
calisma = ayarlar["calisma"]
konu = ayarlar["konu"]
stil = ayarlar["stil"]
slogan_sayisi = ayarlar["slogan_sayisi"]
sicaklik = ayarlar["sicaklik"]
tohum = ayarlar["tohum"]

print(konu, "/", stil)

`konu` Türkçe, çünkü Türkçe slogan sorusuna gidecek. `stil` İngilizce, çünkü görsel yönergesine gidecek (Konu 7).
`ayarlar.json`'u değiştirirsen bu iki hücreyi yeniden çalıştır; yoksa değişkenler eski değerleri taşır.

## Adım 2 — Çalışma klasörü

Hattın bütün çıktıları tek bir klasöre gidecek: `ciktilar/` içinde, adını `ayarlar.json`'daki `calisma` alanından alan
bir klasöre.

- Yol, Konu 8'deki `klasor = "izgara/"` gibi düz bir metin: `"ciktilar/"` + çalışma adı + `"/"`. Sondaki `/` sayesinde
  dosya adını doğrudan ekleyebilirsin: `klasor + "sloganlar.json"`.
- `os.makedirs(klasor, exist_ok=True)` klasörü açar; `ciktilar/` henüz yoksa onu da. `exist_ok=True` "klasör zaten varsa
  hata verme" demek. `os` Python'la birlikte geliyor; Konu 8'deki `os.path.exists` da bu modülden.
- Konu 8'de klasörü `sabirli_ciz` kendisi açıyordu. Bu hatta ilk yazılan dosya bir görsel değil, sloganlar olacak; o yüzden
  klasörü baştan sen açıyorsun.

In [ ]:
import os

klasor = "ciktilar/" + calisma + "/"
os.makedirs(klasor, exist_ok=True)
print(klasor)

Klasörün adında neden otomatik bir tarih yok? Olsaydı her çalıştırmada yeni bir klasör açılırdı; içinde hiçbir şey
"zaten var" olmazdı, hat her şeyi baştan üretirdi. Kural Konu 8'deki gibi: **aynı iş, aynı ad**. Brifi (konu, stil, tohum)
değiştirdiğinde `calisma` adını da değiştir; yeni brif yeni klasörde başlar, eskisi olduğu gibi kalır.

## Adım 3 — Hazır halkalar

`halkalar.py` bu klasördeki yardımcı dosya. İçinde önceki konulardan iki fonksiyon hazır duruyor:

- `modele_sor(soru, sicaklik)`: Konu 5'te yazdığın fonksiyon. Tek farkı: istek başarısız olursa çökmez, `None` verir.
- `sabirli_ciz(yonerge, tohum, yol)`: Konu 8'deki fonksiyon. Görseli en fazla üç kez dener ve `yol`a kendisi kaydeder;
  başarılıysa `yol`u, olmazsa `None` verir.

`halkalar.py` anahtarı `../anahtar.txt`'den kendisi okur. Halkalar hazır; onları **birbirine bağlamak** bugün senin işin.

Aşağıdaki hücrenin çıktısı yok. `FileNotFoundError: ... '../anahtar.txt'` görürsen anahtar dosyası yok ya da yanlış
yerde: `gita3111` klasöründe durmalı (Konu 2).

In [ ]:
from halkalar import modele_sor, sabirli_ciz

Konu 5'teki slogan sorusu burada bir kalıba dönüşüyor: sabit bir kısım, sonunda brifteki konu. Önce tek bir istekle dene.

**Çalıştırmadan önce tahmin et:** model yalnızca sloganı mı yazacak, yoksa başına sonuna bir şey ekleyecek mi?

In [ ]:
soru = "Şu konu için tek bir kısa slogan yaz. Yalnızca sloganı yaz, açıklama ekleme: " + konu

cevap = modele_sor(soru, sicaklik)
print(cevap)

Model talimata her zaman uymaz: sloganı tırnak içine alabilir, başına "Slogan:" yazabilir, sonuna bir açıklama ekleyebilir.
Bu cevap sıradaki halkaya, görsel yönergesine gidecek; o yüzden önce **temizlenmeli**. İki tanıdık araç yeter:
`.strip()` baştaki ve sondaki boşlukları siler (Konu 2'de anahtar dosyasını okurken kullandın), `.replace('"', "")` düz
tırnakları siler (Konu 1'de metni temizlerken).

Üstteki hücre cevap yerine `İstek başarısız, durum kodu: 401` yazdıysa anahtar yanlıştır ve `cevap` `None` olur; aşağıdaki
hücre de `AttributeError: 'NoneType' object has no attribute 'strip'` verir. `anahtar.txt`'yi düzelt, çekirdeği yeniden
başlat (**Restart**), yukarıdan yeniden çalıştır.

In [ ]:
temiz = cevap.strip()
temiz = temiz.replace('"', "")
print(temiz)

Temiz cevapta hâlâ bir fazlalık varsa (kıvrık tırnak “ ”, başta `Slogan:`) aynı kalıpla bir satır daha ekle:
`temiz = temiz.replace("Slogan: ", "")`. Adım 4'teki fonksiyonda aynı iki temizleme satırı var; eklediğin satırı oraya da
yaz (orada değişkenin adı `slogan`).

## Adım 4 — 1. halka: sloganlar (ara çıktıyı sakla)

Bir halkanın sonucu hattın **ara çıktısıdır**. Sloganları bir dosyada saklayacaksın: `sloganlar.json`.
Fonksiyon her çağrıldığında önce şunu sorar: **zaten var mı?**

- Dosya varsa (`os.path.exists`, Konu 8): oku, ver; modele hiç istek gitmez (`return` fonksiyonu orada bitirir).
- Yoksa: `slogan_sayisi` kez sor (`range(slogan_sayisi)`: ayarlarda 3, yani "3 kez"), her cevabı temizleyip listeye ekle,
  listeyi dosyaya yaz, ver.
- Bir cevap gelmezse (`None`) o turu atla (`continue`); hat durmaz.
- Hiç slogan gelmediyse dosyaya yazma. `len(sloganlar) == 0` "liste boş mu?" demek (Konu 5). Boş liste dosyaya girseydi
  sonraki her çağrı onu okur, modele bir daha hiç sormazdı.

Dosyaya yazmak Konu 8'deki günlükle aynı: `json.dump(..., ensure_ascii=False, indent=2)`.

Fonksiyon parametre almıyor; `soru`, `sicaklik` ve `slogan_sayisi`'yı yukarıdaki hücrelerden kullanıyor. Hücre yalnızca
fonksiyonu tanımlar; istekler, fonksiyonu bir sonraki hücrede çağırınca gider. Adım 3'teki deneme sloganı listeye girmez:
fonksiyon üç yeni istek gönderir.

In [ ]:
slogan_dosyasi = klasor + "sloganlar.json"

def slogan_halkasi():
    if os.path.exists(slogan_dosyasi):
        with open(slogan_dosyasi, encoding="utf-8") as dosya:
            sloganlar = json.load(dosya)
        print("Dosyadan okundu, istek gitmedi")
        return sloganlar
    sloganlar = []
    for tur in range(slogan_sayisi):
        cevap = modele_sor(soru, sicaklik)
        if cevap is None:
            print("Slogan gelmedi, atlıyorum")
            continue
        slogan = cevap.strip()
        slogan = slogan.replace('"', "")
        sloganlar.append(slogan)
    if len(sloganlar) == 0:
        print("Hiç slogan gelmedi, dosyaya yazmadım")
        return sloganlar
    with open(slogan_dosyasi, "w", encoding="utf-8") as dosya:
        json.dump(sloganlar, dosya, ensure_ascii=False, indent=2)
    print("Üretildi ve kaydedildi")
    return sloganlar

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

`ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç: üç slogan orada.

Ekranda üç kez `Slogan gelmedi`, ardından `Hiç slogan gelmedi, dosyaya yazmadım` gördüysen istekler başarısız oldu; sebep
çoğunlukla anahtar. `anahtar.txt`'yi düzelt, çekirdeği yeniden başlat (**Restart**), yukarıdan yeniden çalıştır. Dosya
yazılmadığı için sloganlar bu kez gerçekten istenir.

**Tahmin et, sohbete yaz:** aynı çağrıyı bir kez daha yaparsan kaç istek gider? Sloganlar aynı mı kalır?

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

Sloganları saklamanın sebebi yalnızca kota değil, **tutarlılık** da. Sıcaklık 0.9 olduğu için her istekte başka bir slogan
gelir (Konu 5). Sloganlar her çalıştırmada yeniden üretilseydi `gorsel_1.jpg` dünkü sloganla çizilmiş olurdu, günlükte ise
bugünkü slogan yazardı: görsel ile metin birbirini tutmazdı.

Kural: **pahalı ya da her seferinde farklı çıkan** ara çıktıyı sakla.

## Adım 5 — 2. halka: her slogana bir görsel

Her slogan için önce bir görsel yönergesi kur: stil (İngilizce, ayarlardan) + slogan + "görselde yazı olmasın".
Bu da bir kalıp: sabit bir kısım, iki değişen yer (stil ve slogan). Yönergeler saklanmıyor, çünkü sloganlardan ve
ayarlardan her seferinde aynı biçimde, bedavaya kurulur.

In [ ]:
yonergeler = []
for slogan in sloganlar:
    yonerge = stil + ", a poster illustration that expresses this idea: " + slogan + ", no text, no letters"
    yonergeler.append(yonerge)
    print(yonerge)

Sloganlar Türkçe, yönergenin geri kalanı İngilizce. Görsel model çoğunlukla İngilizce metinle eğitildi (Konu 7); Türkçe
sloganın **fikrini** ne kadar anlayacağını birazdan göreceksin. "Görselde yazı olmasın" bilerek orada: görsel modeller
harfleri çoğu zaman bozar, sloganı afişe tasarımcı dizer. Hat yalnızca **taslak** üretir.

Görsel halkası, tek bir görsel için çalışan bir fonksiyon. Slogan halkasının sorusunu o da sorar: zaten var mı? Dosya varsa
(`os.path.exists(yol)`) atlar; dosya orada olduğu için kayıtta durum yine `uretildi` olur. Yoksa görseli `sabirli_ciz` ile
üretir, dosyaya da `sabirli_ciz` kaydeder. Fonksiyon her seferinde bir **kayıt** (sözlük) verir: hangi halka, ne girdi,
hangi dosya çıktı, sonuç ne (`durum`).

- `"gorsel_" + str(sira) + ".jpg"`: `str` sayıyı metne çevirir (Konu 5); metne yalnızca metin eklenebilir.
- `return kayit` fonksiyonu orada bitirir; hangi yoldan çıkarsa çıksın elimize bir kayıt gelir.

In [ ]:
def gorsel_halkasi(sira, yonerge):
    yol = klasor + "gorsel_" + str(sira) + ".jpg"
    kayit = {"halka": "gorsel", "girdi": yonerge, "cikti": yol}
    if os.path.exists(yol):
        kayit["durum"] = "uretildi"
        print("Zaten var, atlıyorum:", yol)
        return kayit
    sonuc = sabirli_ciz(yonerge, tohum, yol)
    if sonuc is None:
        kayit["durum"] = "olmadi"
        print("Olmadı, atlıyorum:", yol)
        return kayit
    kayit["durum"] = "uretildi"
    print("Kaydedildi:", yol)
    return kayit

Şimdi halkayı bir döngüyle her yönergeye uygula; gelen her kayıt listeye eklenir. `sira` her turun **başında** bir artar,
böylece ilk görsel `gorsel_1.jpg` olur. Üç görsel bir dakika kadar sürer.

In [ ]:
kayitlar = []
sira = 0
for yonerge in yonergeler:
    sira = sira + 1
    kayit = gorsel_halkasi(sira, yonerge)
    kayitlar.append(kayit)

print(len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/` klasöründeki görsellere bak. Sohbete yaz:

- Görselde yine de yazı çıktı mı? Harfler okunuyor mu?
- Görseller sloganların **fikrini** mi taşıyor, yoksa yalnızca stili mi? Üç görsel birbirinden ne kadar farklı?

## Adım 6 — Bir halka koparsa hat durmasın

Gerçek bir hatta halkalardan biri er geç kopar: internet gider, kota biter, bir yönerge reddedilir. Bunu beklemek yerine
halkayı bilerek kopar: dördüncü görseli **boş bir yönergeyle** iste. Görsel model boş yönergeyi kabul etmez.

**Tahmin et:** hücre hata verip duracak mı? Kayıtta ne yazacak? (Üç denemeyle 10 saniye kadar sürer.)

In [ ]:
kayit = gorsel_halkasi(4, "")
kayitlar.append(kayit)

print(kayit)
print(len(kayitlar), "kayıt")

Sunucu üç kez de reddetti (durum kodu 400). Ama hücre çökmedi: `gorsel_halkasi` hata yerine `"durum": "olmadi"` yazan bir
kayıt verdi. Bu çağrı bir döngünün içinde olsaydı döngü sıradaki görsele geçerdi. Hat iki şeyi birlikte yapıyor:

1. Kopan halkayı **kayda geçiriyor** (hangi görsel, `durum`: `olmadi`).
2. Kopmayanlarla **devam ediyor**.

`gorsel_4.jpg` hiç oluşmadı; hattı yarın yeniden çalıştırırsan yalnızca o denenir.

Bu hücreyi iki kez çalıştırırsan aynı kayıt listeye iki kez girer (`5 kayıt`). Listeyi düzeltmek için Adım 5'in döngü
hücresinden yeniden başla: görseller zaten var, istek gitmez.

## Adım 7 — Üretim günlüğü: `uretim.json`

Hattın bütün kayıtları tek bir dosyada, **üretim günlüğünde** toplanır: hangi ayarlarla çalıştı, hangi sloganlar çıktı,
her görselin sonucu ne oldu. Konu 8'deki günlüğe benziyor; farkı, bu kez bir sözlüğün içinde üç bölüm olması.

In [ ]:
gunluk = {"ayarlar": ayarlar, "sloganlar": sloganlar, "kayitlar": kayitlar}

with open(klasor + "uretim.json", "w", encoding="utf-8") as dosya:
    json.dump(gunluk, dosya, ensure_ascii=False, indent=2)

print("Günlük yazıldı:", len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/uretim.json`'u VS Code'da aç. `durum`u `olmadi` olan kayıtlar eksiklerin listesi: hattı yeniden
çalıştırınca denenecek olanlar. Final projesinin teslimindeki **üretim günlüğü** bu dosya: hangi brif, hangi model ayarı,
ne çıktı, ne eksik kaldı.

**Defteri yarın baştan çalıştırırsan ne görürsün?** Adım 3 yine bir deneme isteği gönderir. Adım 4'teki iki çağrıda da
`Dosyadan okundu, istek gitmedi`, Adım 5'te üç kez `Zaten var, atlıyorum` görürsün. Adım 6'daki istek yine üç kez reddedilir:
`gorsel_4.jpg` hiç oluşmadığı için her seferinde yeniden denenir. Reddedilenleri saymazsan giden tek istek Adım 3'teki deneme.
Hat kaldığı yerden devam ediyor: biten iş yeniden üretilmiyor, yalnızca eksik olan deneniyor.

## Bonus

**1. Pano: slogan ve görsel yan yana.** Konu 8'deki kontak sayfasının tek satırlık hâli: her kutuda bir görsel, üstünde onu
doğuran slogan. `plt.imread` görseli okur, `plt.imshow` kutuya koyar; dosya yoksa kutu boş kalır.

In [ ]:
import matplotlib.pyplot as plt

sutun = len(sloganlar)

plt.figure(figsize=(12, 5))
sira = 0
for slogan in sloganlar:
    sira = sira + 1
    yol = klasor + "gorsel_" + str(sira) + ".jpg"
    plt.subplot(1, sutun, sira)
    if os.path.exists(yol):
        resim = plt.imread(yol)
        plt.imshow(resim)
    plt.title(slogan, fontsize=9)
    plt.axis("off")

plt.savefig(klasor + "pano.png", bbox_inches="tight")
plt.show()

Panoya tasarımcı gözüyle bak. Hangi slogan hangi görselle güçlü? Slogan ile görselin **aynı şeyi söylediği** kutu mu daha iyi,
yoksa birbirini **tamamladığı** kutu mu (görsel bir şey gösteriyor, slogan ona yeni bir anlam ekliyor)? Sohbete tek cümle yaz.

**2. Ara çıktıya elle dokun.** Ara çıktıyı saklamanın bir faydası daha var: hattın ortasına girip elle düzeltme yapabilirsin.

1. `ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç, ikinci sloganı kendi yazdığın bir sloganla değiştir, kaydet.
   (Tırnakları ve virgülleri bırak; JSON biçimi bozulmasın.)
2. Aynı klasördeki `gorsel_2.jpg`'yi sil.
3. Adım 4'ün ilk çağrı hücresinden (`sloganlar = slogan_halkasi()`) Adım 5'in sonuna kadar sırayla yeniden çalıştır,
   sonra Adım 7'nin günlük hücresini ve panoyu.

Kaç istek gitti? Pano neden yalnızca bir kutuda değişti?

## Sıradaki konu

Bu hatta adımların sırasını sen belirledin. Konu 11'de modele bilmediği bir belgeyi okutacağız, Konu 12'de sırayı model belirleyecek.